# V2 Housing Supply Features

## 목적

서울 25개 구의 **주택 공급 흐름과 미분양**을 구×월 Feature로 만든다.
모든 값은 **그 시점에 공개돼 있던 정보만** 쓴다.

### 주요 데이터

| 데이터 | 출처 | 단위 | 공개 시점 처리 |
|---|---|---|---|
| 주택인허가 원장 (승인·사용검사) | 국토부 건축HUB 주택인허가정보 API (`04a_collect_housing_permit.py`, 2026-10-08 수집) | 대지(단지) | 사건월 M → **M+2월 행부터** |
| 미분양·준공후 미분양 | 국토부 통계누리 월별 파일 92개 (2019-01 ~ 2026-08, `04b_download_unsold.py`) | 구 | M월 말 기준 → M+1월 말 공표 → **M+2월 행부터**, 각 달 파일의 최초 공표값 |
| 서울 아파트 인허가·준공 공식 월별 실적 (2019-01 ~ 2026-08) | 국토부 주택건설실적 (통계누리 월별 파일, `04c_download_official_supply.py`) | 시도 | 각 달 파일의 **최초 공표값**, M+2월 행부터. 원장 검증에도 사용 |
| 아파트 입주예정물량 (2026-08-31 공개분) | 한국부동산원 | 단지 | **최신 회차만 존재 → 웹 표시용만** |

### 조사에서 확인한 제약

- 국토부 인허가·착공·준공 통계는 **시도 단위까지만** 공표된다. 구 단위는 원장을 직접 집계해야 한다.
- 원장의 **착공일은 2023년 이후 거의 기록되지 않는다** → 착공 Feature와 '착공 후 미준공' 입주예정 재구성은 쓰지 않는다.
- 원장의 **사용검사 예정일**은 현재 값만 있고 사후 수정됐을 수 있다 → 모델 Feature로 쓰지 않는다.
- 원장은 현재 snapshot이라 과거에 이미 입력돼 있었는지 알 수 없다. 사업계획승인·사용검사는 공개 행정행위이므로
  사건일 기준으로 쓰되, 입력 지연을 고려해 2개월 늦춘다. 2022-08 이후 기록은 생성일(`crtnDay`)로 지연을 점검한다.
- 입주예정물량은 과거 회차가 남아 있지 않아 모델에 쓰면 누수가 된다. 지금부터 회차마다 쌓는다.

In [1]:
# ============================================================
# 경로·상수
# ============================================================

from pathlib import Path
import glob
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="The 'generic' unit for NumPy timedelta")
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("프로젝트 루트에서 data 폴더를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"
SUPPLY_DIR = PROCESSED_DIR / "supply"
SUPPLY_DIR.mkdir(parents=True, exist_ok=True)

PERMIT_PATH = EXTERNAL_DIR / "housing_permit_basis_seoul_20261008.parquet"
UNSOLD_RAW_DIR = EXTERNAL_DIR / "unsold_raw"
UNSOLD_CACHE = SUPPLY_DIR / "unsold_seoul_gu_monthly.parquet"
OFFICIAL_2021 = EXTERNAL_DIR / "supply_official" / "molit_permit_202112.xls"
IPJU_PATH = EXTERNAL_DIR / "ipju" / "ipju_20260831.xlsx"
OFFICIAL_SERIES = EXTERNAL_DIR / "supply_official" / "molit_seoul_permit_completion.csv"  # 04c로 추출

PUBLISH_LAG = 2               # 사건월 M → M+2월 행부터 사용
PIPELINE_YEARS = 6            # 승인 후 6년 안에 준공 기록이 없으면 파이프라인에서 제외
LATEST_DATE = pd.Timestamp("2026-10-08")

region_month = pd.read_parquet(PROCESSED_DIR / "market_leading_v2.parquet", columns=["기준월", "자치구", "권역"])
ALL_GU = sorted(region_month["자치구"].unique())

print("PROJECT_DIR:", PROJECT_DIR)
print("구×월 Grid:", region_month.shape)

PROJECT_DIR: c:\Users\uk\Apartment price
구×월 Grid: (1500, 3)


## 1. 주택인허가 원장

In [2]:
# ============================================================
# 원장 로드: 세대수가 있는 공동주택 기록만
# - 원장 13만 건 중 대부분은 세대수 0인 부속 기록이다.
# ============================================================

permit_raw = pd.read_parquet(PERMIT_PATH)


def to_date(s):
    s = s.astype(str).str.strip()
    return pd.to_datetime(s.where(s.str.match(r"^\d{8}$")), format="%Y%m%d", errors="coerce")


permit = permit_raw.copy()
permit["세대수"] = pd.to_numeric(permit["totHhldCnt"], errors="coerce").fillna(0)
for col, name in [("apprvDay", "승인일"), ("stcnsDay", "착공일"), ("useInsptDay", "사용검사일"),
                  ("useInsptSchedDay", "사용검사예정일"), ("crtnDay", "생성일")]:
    permit[name] = to_date(permit[col])

permit["대지"] = (
    permit["sigunguCd"] + permit["bjdongCd"] + permit["platGbCd"] + permit["bun"] + permit["ji"]
    + permit["block"].str.strip() + permit["lot"].str.strip()
)

housing = permit[
    permit["세대수"].gt(0)
    & permit["purpsCdNm"].str.strip().eq("공동주택")
    & permit["승인일"].notna()
].copy()

print("원장 전체:", permit_raw.shape)
print("세대수>0 공동주택:", housing.shape, "세대 합", int(housing["세대수"].sum()))
print("자치구 수:", housing["자치구"].nunique())

원장 전체: (130205, 32)
세대수>0 공동주택: (5054, 39) 세대 합 3167984
자치구 수: 25


In [3]:
# ============================================================
# 기록 품질: 승인연도별 착공일·사용검사일 결측
# - 착공일은 2023년 이후 승인 건에서 90% 이상 비어 있다 → 착공 Feature 제외
# ============================================================

quality = housing.groupby(housing["승인일"].dt.year).agg(
    기록수=("세대수", "size"),
    승인세대=("세대수", "sum"),
    착공일_결측=("착공일", lambda s: s.isna().mean()),
    사용검사일_결측=("사용검사일", lambda s: s.isna().mean()),
)
display(quality.loc[2015:2026].round(2))

display(
    pd.DataFrame({
        "착공 세대": housing.groupby(housing["착공일"].dt.year)["세대수"].sum(),
        "사용검사 세대": housing.groupby(housing["사용검사일"].dt.year)["세대수"].sum(),
    }).loc[2017:2026].fillna(0).astype(int)
)

,기록수,승인세대,착공일_결측,사용검사일_결측
승인일,,,,
2015,94,55612,0.17,0.22
2016,71,32066,0.24,0.25
2017,147,86797,0.29,0.32
2018,97,39294,0.38,0.39
2019,89,37414,0.45,0.46
2020,100,31516,0.42,0.43
2021,160,66676,0.60,0.61
2022,77,21675,0.68,0.68
2023,77,36180,0.90,0.90


,착공 세대,사용검사 세대
2017.0,55350,25214
2018.0,48813,35019
2019.0,49760,49304
2020.0,41550,42734
2021.0,34943,41962
2022.0,28729,25629
2023.0,7474,18878
2024.0,689,18813
2025.0,319,59184
2026.0,0,24160


In [4]:
# ============================================================
# 중복 제거
# - 같은 대지·같은 승인일·같은 세대수 기록이 관리번호만 다르게 2건씩 있다(구 시스템 번호 + 신 번호).
# - 승인 사건: (대지, 승인일, 세대수) 중복 제거
# - 사용검사 사건: (대지, 사용검사일, 세대수) 중복 제거
# - 중복 중에서는 생성일이 가장 이른 기록을 남긴다 (신 관리번호로 다시 만든 기록은 생성일이 늦다)
# ============================================================

approvals = (
    housing.sort_values("생성일", kind="stable")
    .drop_duplicates(["대지", "승인일", "세대수"])
    [["대지", "자치구", "승인일", "세대수", "생성일", "bldNm", "platPlc"]]
    .copy()
)

completions = (
    housing[housing["사용검사일"].notna()]
    .sort_values("생성일", kind="stable")
    .drop_duplicates(["대지", "사용검사일", "세대수"])
    [["대지", "자치구", "사용검사일", "세대수", "생성일", "bldNm", "platPlc"]]
    .copy()
)

print("승인 사건:", len(approvals), "(원본", len(housing), ")")
print("사용검사 사건:", len(completions))

display(
    pd.DataFrame({
        "원본": housing.groupby(housing["승인일"].dt.year)["세대수"].sum(),
        "중복 제거": approvals.groupby(approvals["승인일"].dt.year)["세대수"].sum(),
    }).loc[2017:2026].astype(int)
)

승인 사건: 4787 (원본 5054 )
사용검사 사건: 2637


,원본,중복 제거
승인일,,
2017,86797,76730
2018,39294,33058
2019,37414,33338
2020,31516,27115
2021,66676,53705
2022,21675,16610
2023,36180,31186
2024,51173,42202
2025,38181,35824


In [5]:
# ============================================================
# 공식 통계와 비교 (2021년 서울 누계, 국토부 주택건설실적)
# - 공식 아파트에는 건축허가 아파트(주상복합 등)가 포함되고, 원장 공동주택에는 주택법 승인 연립·다세대가 포함된다.
# ============================================================

official = pd.read_excel(OFFICIAL_2021, sheet_name="참고1 주택유형별 대분류", header=None)
seoul_row = official[official[0].astype(str).str.strip().eq("서울")].iloc[0]
official_total, official_apt = int(seoul_row[1]), int(seoul_row[5])

ledger_2021 = int(approvals.loc[approvals["승인일"].dt.year == 2021, "세대수"].sum())
ledger_2021_raw = int(housing.loc[housing["승인일"].dt.year == 2021, "세대수"].sum())

print(f"공식 2021 서울 인허가: 전체 {official_total:,}호 / 아파트 {official_apt:,}호")
print(f"원장 2021 승인 세대: 원본 {ledger_2021_raw:,} → 중복 제거 {ledger_2021:,} ({ledger_2021 / official_apt - 1:+.1%} vs 공식 아파트)")

assert abs(ledger_2021 / official_apt - 1) < 0.05

공식 2021 서울 인허가: 전체 81,637호 / 아파트 52,819호
원장 2021 승인 세대: 원본 66,676 → 중복 제거 53,705 (+1.7% vs 공식 아파트)


In [6]:
# ============================================================
# 공식 통계와 다년 비교 (2019~2025, 서울 아파트)
# - 공표치: 각 해 12월 파일의 누계 / 월별: 각 달 파일의 최초 공표 월계
# ============================================================

official_series = pd.read_csv(OFFICIAL_SERIES, encoding="utf-8-sig")

official_monthly = official_series[official_series["basis"].eq("monthly_file")].copy()
official_monthly["ref_month"] = pd.PeriodIndex(
    official_monthly["year"].astype(int).astype(str) + "-" + official_monthly["month"].astype(int).astype(str), freq="M"
)
official_monthly = official_monthly[["kind", "ref_month", "apartment", "total"]].sort_values(["kind", "ref_month"])
assert not official_monthly.duplicated(["kind", "ref_month"]).any()

annual = official_series[official_series["basis"].eq("dec_file_ytd")].set_index(["kind", "year"])["apartment"]
compare_years = pd.DataFrame({
    "원장 승인": approvals.groupby(approvals["승인일"].dt.year)["세대수"].sum(),
    "공식 인허가 아파트": annual["인허가"],
    "원장 준공": completions.groupby(completions["사용검사일"].dt.year)["세대수"].sum(),
    "공식 준공 아파트": annual["준공"],
}).loc[2019:2025]
compare_years["승인/공식"] = compare_years["원장 승인"] / compare_years["공식 인허가 아파트"]
compare_years["준공/공식"] = compare_years["원장 준공"] / compare_years["공식 준공 아파트"]
display(compare_years.round(2))

for kind, ledger, col in [("인허가", approvals, "승인일"), ("준공", completions, "사용검사일")]:
    led = ledger.groupby(ledger[col].dt.to_period("M"))["세대수"].sum()
    off = official_monthly[official_monthly["kind"].eq(kind)].set_index("ref_month")["apartment"]
    j = pd.concat([off, led], axis=1, keys=["공식", "원장"]).fillna(0).loc["2019-01":"2026-08"]
    r12 = j.rolling(12).sum().dropna()
    print(f"{kind}: 월별 12개월 합 상관 {r12.corr().iloc[0, 1]:.2f}")

,원장 승인,공식 인허가 아파트,원장 준공,공식 준공 아파트,승인/공식,준공/공식
2019,33338,37767.0,47599.0,45630.0,0.88,1.04
2020,27115,30404.0,42734.0,56784.0,0.89,0.75
2021,53705,52819.0,41962.0,46651.0,1.02,0.90
2022,16610,25133.0,25629.0,31836.0,0.66,0.81
2023,31186,34965.0,18878.0,38028.0,0.89,0.50
2024,42202,47632.0,18813.0,32988.0,0.89,0.57
2025,35824,35512.0,59184.0,41893.0,1.01,1.41


인허가: 월별 12개월 합 상관 0.95
준공: 월별 12개월 합 상관 0.62


**결정:** 원장의 준공 기록은 공식 통계와 맞지 않는다 (2023·2024년 약 0.5배, 2025년 1.4배, 12개월 합 상관 약 0.6).
Valid·Test 구간과 겹치므로 **원장 기반 준공은 모델에서 제외**하고,
서울 단위는 공식 월별 실적(최초 공표값)으로, 구 단위 준공은 건축물대장(사용승인일)으로 대체한다(`04e`, 검증 후 반영).
원장 승인은 공식 통계와 잘 맞지만(상관 약 0.95) 4장의 입력 지연 문제로 모델에서 제외한다.

In [7]:
# ============================================================
# 입력 지연 점검: 2022-09 이후 생성된 기록의 (생성일 - 사건일)
# - 2022-08-13은 시스템 이전일이라 그 전 기록은 지연을 알 수 없다.
# ============================================================

def entry_lag(df, event_col):
    recent = df[df["생성일"] >= "2022-09-01"].copy()
    lag = (recent["생성일"] - recent[event_col]).dt.days
    return pd.Series({
        "기록수": len(recent),
        "중위 지연(일)": lag.median(),
        "60일 초과 비율": (lag > 60).mean(),
        "60일 초과 세대 비율": recent.loc[lag > 60, "세대수"].sum() / recent["세대수"].sum(),
    })


display(pd.DataFrame({
    "승인": entry_lag(approvals[approvals["승인일"] >= "2022-09-01"], "승인일"),
    "사용검사": entry_lag(completions[completions["사용검사일"] >= "2022-09-01"], "사용검사일"),
}).round(3))

,승인,사용검사
기록수,332.000,222.000
중위 지연(일),3.000,0.000
60일 초과 비율,0.476,0.036
60일 초과 세대 비율,0.312,0.069


## 2. 미분양 (구별, 최초 공표값)

각 달 파일의 '시군구별' 시트 마지막 열(그 달 값)과 '공사완료후' 시트를 읽는다.
나중 파일로 덮어쓰지 않고 **그 달에 처음 공표된 값**을 쓴다.

In [8]:
# ============================================================
# 미분양 파싱 (92개 파일, 처음 한 번만 실행하고 캐시)
# - '시군구별' 시트는 마지막 열이 그 달 값이다 (머리글은 오타가 있어 쓰지 않는다: 2019-10 파일의 '19.9 등).
# - 2019~2021년 파일은 0을 '-' 또는 빈칸으로 표기한다 → 0 (서울 계 = 25개 구 합으로 검증)
# - 그 달 파일의 마지막 열이 통째로 비어 있으면(2026-02) 다음 달 파일의 끝에서 두 번째 열을 쓰고 표시한다.
# ============================================================

def seoul_rows(sheet):
    d = sheet.copy()
    d[0] = d[0].ffill()
    d = d[d[0].astype(str).str.replace(" ", "").str.contains("서울", na=False)].copy()
    d["구"] = d[1].astype(str).str.replace(" ", "")
    return d


def to_number(s):
    return pd.to_numeric(s.astype(str).str.strip().replace({"-": "0", "nan": "0", "": "0"}), errors="coerce")


def read_month(path, offset=0):
    book = pd.ExcelFile(path)
    sheet = pd.read_excel(book, "시군구별★", header=None)
    col = sheet.columns[-1 - offset]
    rows = seoul_rows(sheet)
    raw = rows[col]
    return rows["구"].to_numpy(), raw.notna().any(), to_number(raw).to_numpy(), book


def parse_unsold(ref, files):
    gu, has_value, values, book = read_month(files[ref])
    source = "당월 파일"
    if not has_value and (ref + 1) in files:
        gu, has_value, values, _ = read_month(files[ref + 1], offset=1)
        source = "다음 달 파일"

    done = seoul_rows(pd.read_excel(book, "공사완료후★", header=None))
    return pd.DataFrame({"ref_month": str(ref), "구": gu, "미분양": values, "미분양_출처": source}).merge(
        pd.DataFrame({"구": done["구"].to_numpy(), "준공후미분양": to_number(done[2]).to_numpy()}),
        on="구", how="outer",
    )


if UNSOLD_CACHE.exists():
    unsold = pd.read_parquet(UNSOLD_CACHE)
else:
    files = {
        pd.Period(re.search(r"unsold_(\d{6})", p.name).group(1), freq="M"): p
        for p in sorted(UNSOLD_RAW_DIR.glob("unsold_*.xlsx"))
    }
    unsold = pd.concat([parse_unsold(ref, files) for ref in sorted(files)], ignore_index=True)
    unsold.to_parquet(UNSOLD_CACHE, index=False)

unsold["ref_month"] = pd.PeriodIndex(unsold["ref_month"], freq="M")
print(unsold.shape, unsold["ref_month"].min(), "~", unsold["ref_month"].max())
print("다음 달 파일로 대체한 달:", sorted(unsold.loc[unsold["미분양_출처"].ne("당월 파일"), "ref_month"].astype(str).unique()))

(2392, 5) 2019-01 ~ 2026-08
다음 달 파일로 대체한 달: ['2026-02']


In [9]:
# ============================================================
# 미분양 검증: 92개월 × (계 + 25개 구), 서울 계 = 25개 구 합
# ============================================================

assert unsold["ref_month"].nunique() == 92
assert set(unsold["구"]) == set(ALL_GU) | {"계"}
assert unsold.groupby("ref_month").size().eq(26).all()
assert unsold[["미분양", "준공후미분양"]].notna().all().all()

wide = unsold.pivot_table(index="ref_month", columns="구", values=["미분양", "준공후미분양"], aggfunc="first")
for kind in ["미분양", "준공후미분양"]:
    gap = (wide[kind][ALL_GU].sum(axis=1) - wide[kind]["계"]).abs()
    print(f"{kind}: 서울 계와 구 합 차이 최대 {gap.max():.0f}")
    assert gap.max() == 0

seoul_unsold = wide.xs("계", axis=1, level=1)
display(seoul_unsold.loc[["2019-12", "2020-12", "2021-12", "2022-12", "2023-12", "2024-12", "2025-12", "2026-08"]])

zero_share = (wide["미분양"][ALL_GU] == 0).mean().mean()
print(f"구×월 미분양 0 비율: {zero_share:.1%}")

미분양: 서울 계와 구 합 차이 최대 0
준공후미분양: 서울 계와 구 합 차이 최대 0


,미분양,준공후미분양
ref_month,,
2019-12,151,141
2020-12,49,48
2021-12,54,52
2022-12,953,340
2023-12,958,461
2024-12,957,633
2025-12,939,673
2026-08,898,674


구×월 미분양 0 비율: 64.3%


## 3. 구×월 Feature (`supply_v2.parquet`)

기준월 m 행은 **m-2월까지의 사건**만 쓴다 (사건월 M → M+2월 행부터).

| Feature | 정의 | 용도 |
|---|---|---|
| `인허가아파트_서울_6m/12m`, `준공아파트_서울_3m/6m/12m` | 국토부 공식 서울 아파트 월별 실적(최초 공표값) 합, m-2월까지 | 모델 |
| `미분양_구`, `준공후미분양_구`, `미분양_있음` | m-2월 말 기준 값 (m-1월 말 공표) | 모델 |
| `미분양_서울`, `준공후미분양_서울`, `미분양_서울_3m변화` | 서울 합계 | 모델 |
| `준공세대_3m/6m/12m`, `준공12m_대비5년평균`, `준공세대_12m_서울` | 원장 사용검사(준공) 세대 합 | 검증용 (공식 통계와 불일치로 제외) |
| `승인세대_6m/12m`, `승인12m_대비5년평균`, `승인세대_12m_서울` | 사업계획승인 세대 합 | 검증용 (4장 결과로 모델 제외) |
| `공급파이프라인_세대` | m-2월 말까지 승인됐고 아직 준공 기록이 없는 세대 (승인 후 6년 이내) | 검증용 (모델 제외) |

In [10]:
# ============================================================
# 월별 패널 (구 × 월)
# ============================================================

PANEL_MONTHS = pd.period_range("2005-01", "2026-06", freq="M")


def monthly_sum(events, date_col, months=PANEL_MONTHS):
    m = events.assign(month=events[date_col].dt.to_period("M"))
    s = m.groupby(["자치구", "month"])["세대수"].sum()
    idx = pd.MultiIndex.from_product([ALL_GU, months], names=["자치구", "month"])
    return s.reindex(idx, fill_value=0)


def rolling_past(s, window, lag=PUBLISH_LAG):
    # m월 값 = (m-lag-window+1) ~ (m-lag)월 합
    return s.groupby(level="자치구").transform(lambda x: x.rolling(window, min_periods=window).sum().shift(lag))


def pipeline_stock(approvals, completions, months, lag=PUBLISH_LAG, years=PIPELINE_YEARS):
    # 기준월마다 그 시점까지의 기록만으로 대지별 최초 승인월·최신 세대수·최초 준공월을 계산한다.
    # (전체 기록의 최대 세대수를 쓰면 이후 변경승인이 과거 값에 섞이는 누수가 된다)
    # 같은 날 세대수가 다른 기록이 있어 정렬을 고정한다 (같은 날이면 큰 세대수가 최신)
    a = (approvals.assign(승인월=approvals["승인일"].dt.to_period("M"))
         .sort_values(["승인일", "세대수", "대지"], kind="stable"))
    c = completions.assign(준공월=completions["사용검사일"].dt.to_period("M"))

    rows = []
    for m in months:
        cut = m - lag
        known = a[a["승인월"] <= cut]
        sites = known.groupby(["대지", "자치구"]).agg(승인월=("승인월", "min"), 세대수=("세대수", "last")).reset_index()
        done = c[c["준공월"] <= cut].groupby("대지")["준공월"].min()
        alive = (sites["승인월"] > cut - 12 * years) & ~sites["대지"].isin(done.index)
        rows.append(sites[alive].groupby("자치구")["세대수"].sum().rename(m))

    out = pd.concat(rows, axis=1).reindex(ALL_GU).fillna(0)
    return out.stack().rename_axis(["자치구", "month"])


def official_features(official, months, lag=PUBLISH_LAG):
    # 서울 아파트 인허가·준공 공식 월별 실적(최초 공표값)의 3·6·12개월 합, m월 행에 (m-lag)월까지
    out = pd.DataFrame({"month": months})
    for kind, name, windows in [("인허가", "인허가아파트_서울", (6, 12)), ("준공", "준공아파트_서울", (3, 6, 12))]:
        s = official[official["kind"].eq(kind)].set_index("ref_month")["apartment"].sort_index()
        s = s.reindex(pd.period_range(s.index.min(), s.index.max(), freq="M"))
        for w in windows:
            r = s.rolling(w, min_periods=w).sum()
            r.index = r.index + lag
            out[f"{name}_{w}m"] = out["month"].map(r)
    return out


def build_supply(approvals, completions, unsold, months, lag=PUBLISH_LAG, official=None):
    official = official_monthly if official is None else official
    a = monthly_sum(approvals, "승인일")
    c = monthly_sum(completions, "사용검사일")

    f = pd.DataFrame(index=a.index)
    f["승인세대_6m"] = rolling_past(a, 6, lag)
    f["승인세대_12m"] = rolling_past(a, 12, lag)
    f["준공세대_3m"] = rolling_past(c, 3, lag)
    f["준공세대_6m"] = rolling_past(c, 6, lag)
    f["준공세대_12m"] = rolling_past(c, 12, lag)

    # 같은 구 직전 60개월 평균 대비 (현재 값을 포함하지 않도록 한 달 더 shift)
    for col, name in [("승인세대_12m", "승인12m_대비5년평균"), ("준공세대_12m", "준공12m_대비5년평균")]:
        base = f[col].groupby(level="자치구").transform(lambda x: x.shift(1).rolling(60, min_periods=36).mean())
        f[name] = f[col] / base.replace(0, np.nan)

    f = f.reset_index()
    f = f[f["month"].isin(months)]

    f = f.merge(pipeline_stock(approvals, completions, months, lag).rename("공급파이프라인_세대").reset_index(),
                on=["자치구", "month"], how="left")

    seoul = f.groupby("month")[["승인세대_12m", "준공세대_12m"]].sum().add_suffix("_서울").reset_index()
    f = f.merge(seoul, on="month", how="left")

    # 미분양: m월 행에 (m-lag)월 기준 값
    u = unsold.copy()
    u["month"] = u["ref_month"] + lag
    gu = u[u["구"].ne("계")].rename(columns={"구": "자치구", "미분양": "미분양_구", "준공후미분양": "준공후미분양_구"})
    se = u[u["구"].eq("계")].rename(columns={"미분양": "미분양_서울", "준공후미분양": "준공후미분양_서울"})
    se = se.sort_values("month")
    se["미분양_서울_3m변화"] = se["미분양_서울"] - se["미분양_서울"].shift(3)

    f = f.merge(gu[["자치구", "month", "미분양_구", "준공후미분양_구"]], on=["자치구", "month"], how="left")
    f = f.merge(se[["month", "미분양_서울", "준공후미분양_서울", "미분양_서울_3m변화"]], on="month", how="left")
    f["미분양_있음"] = (f["미분양_구"] > 0).astype(int)
    f = f.merge(official_features(official, months, lag), on="month", how="left")
    return f


MODEL_MONTHS = pd.PeriodIndex(region_month["기준월"].unique(), freq="M")
supply_rm = build_supply(approvals, completions, unsold, MODEL_MONTHS)

SUPPLY_FEATURES = [
    "승인세대_6m", "승인세대_12m", "준공세대_3m", "준공세대_6m", "준공세대_12m",
    "공급파이프라인_세대", "승인12m_대비5년평균", "준공12m_대비5년평균",
    "미분양_구", "준공후미분양_구", "미분양_있음",
    "미분양_서울", "준공후미분양_서울", "미분양_서울_3m변화",
    "승인세대_12m_서울", "준공세대_12m_서울",
    "인허가아파트_서울_6m", "인허가아파트_서울_12m",
    "준공아파트_서울_3m", "준공아파트_서울_6m", "준공아파트_서울_12m",
]

# 원장 승인·파이프라인: 입력 지연(4장)으로 제외 / 원장 준공: 공식 통계와 불일치(1장)로 제외
# 구 단위 준공은 건축물대장(04e) 검증 후 추가한다.
MODEL_SUPPLY_FEATURES = [
    "인허가아파트_서울_6m", "인허가아파트_서울_12m",
    "준공아파트_서울_3m", "준공아파트_서울_6m", "준공아파트_서울_12m",
    "미분양_구", "준공후미분양_구", "미분양_있음",
    "미분양_서울", "준공후미분양_서울", "미분양_서울_3m변화",
]

print(supply_rm.shape)
display(supply_rm[SUPPLY_FEATURES].describe().T.round(2))

(1500, 23)


,count,mean,std,min,25%,50%,75%,max
승인세대_6m,1500.0,705.70,1181.98,0.0,0.00,191.00,868.00,10288.00
승인세대_12m,1500.0,1444.34,1791.23,0.0,205.00,825.00,1993.00,14313.00
준공세대_3m,1500.0,316.79,890.57,0.0,0.00,0.00,211.25,13237.00
준공세대_6m,1500.0,638.78,1297.92,0.0,0.00,88.00,795.00,13267.00
준공세대_12m,1500.0,1244.02,1763.59,0.0,74.00,729.00,1757.75,14371.00
공급파이프라인_세대,1500.0,6696.69,4930.87,112.0,3409.00,5863.00,8607.00,25683.00
승인12m_대비5년평균,1500.0,1.19,1.64,0.0,0.25,0.72,1.56,17.84
준공12m_대비5년평균,1500.0,0.95,1.31,0.0,0.10,0.53,1.31,16.35
미분양_구,1500.0,32.33,73.05,0.0,0.00,0.00,36.00,1156.00
준공후미분양_구,1500.0,15.79,46.81,0.0,0.00,0.00,1.00,333.00


In [11]:
# ============================================================
# 예시: 서울 합계 흐름과 대규모 공급 구
# ============================================================

display(
    supply_rm.groupby("month")[["승인세대_12m_서울", "준공세대_12m_서울", "미분양_서울", "준공후미분양_서울"]]
    .first()
    .loc[lambda d: d.index.month.isin([1, 7])]
)

latest = supply_rm[supply_rm["month"] == MODEL_MONTHS.max()].set_index("자치구")
display(latest[["승인세대_12m", "준공세대_12m", "공급파이프라인_세대", "미분양_구"]]
        .sort_values("공급파이프라인_세대", ascending=False).head(8))

,승인세대_12m_서울,준공세대_12m_서울,미분양_서울,준공후미분양_서울
month,,,,
2021-07,43792.0,44843.0,71,70
2022-01,57298.0,40198.0,54,52
2022-07,37732.0,32698.0,688,37
2023-01,17335.0,25204.0,865,209
2023-07,21389.0,22391.0,1144,367
2024-01,30861.0,24118.0,877,401
2024-07,30269.0,21399.0,974,539
2025-01,35106.0,17310.0,931,603
2025-07,48442.0,23410.0,989,692


,승인세대_12m,준공세대_12m,공급파이프라인_세대,미분양_구
자치구,,,,
송파구,4309.0,4668.0,15978,0
동작구,1215.0,2256.0,14686,0
노원구,211.0,742.0,13776,0
영등포구,5858.0,925.0,12318,0
용산구,2331.0,0.0,9927,1
동대문구,421.0,5834.0,9144,57
강남구,0.0,8245.0,8156,0
관악구,2857.0,154.0,7023,4


## 4. 누수 검증

1. **재계산 비교:** 각 기준월 m에 대해 m-2월 이후 사건을 모두 지우고 다시 계산한 값이 전체 데이터로 계산한 값과 같아야 한다.
2. **검증 자체 점검:** 지연을 1개월로 줄이면(m-1월 사건 포함) 위 비교가 차이를 잡아내야 한다.
3. **입력 지연 민감도:** 2022-10 이후 기준월에 대해, 기준일까지 생성된 기록만 쓴 값과 비교한다(원장 snapshot의 사후 입력 영향).

In [12]:
# ============================================================
# 1·2. 재계산 비교 + 검증 자체 점검
# ============================================================

def truncated(m, lag):
    cut = m - lag
    a = approvals[approvals["승인일"].dt.to_period("M") <= cut]
    c = completions[completions["사용검사일"].dt.to_period("M") <= cut]
    u = unsold[unsold["ref_month"] <= cut]
    o = official_monthly[official_monthly["ref_month"] <= cut]
    return a, c, u, o


def compare(lag_used, lag_truth=PUBLISH_LAG, months=MODEL_MONTHS):
    full = build_supply(approvals, completions, unsold, months, lag=lag_used).set_index(["month", "자치구"])
    bad, cols = 0, set()
    for m in months:
        a, c, u, o = truncated(m, lag_truth)
        sub = build_supply(a, c, u, pd.PeriodIndex([m]), lag=lag_used, official=o).set_index(["month", "자치구"])
        diff = sub[SUPPLY_FEATURES].compare(full.loc[sub.index, SUPPLY_FEATURES])
        if not diff.empty:
            bad += 1
            cols |= set(diff.columns.get_level_values(0))
    print(f"  (lag={lag_used}) 차이 난 Feature:", sorted(cols) or "없음")
    return bad


leak_months = compare(PUBLISH_LAG)
print("누수 의심 기준월:", leak_months)
assert leak_months == 0

probe_months = MODEL_MONTHS[::6]
detected = compare(1, months=probe_months)
print(f"지연 1개월로 줄였을 때 차이가 난 기준월: {detected} / {len(probe_months)}")
assert detected > 0

print("✅ 공급 Feature: 기준월 m-2월 이후 사건 미사용, 검증이 실제 누수를 잡아냄")

  (lag=2) 차이 난 Feature: 없음
누수 의심 기준월: 0


  (lag=1) 차이 난 Feature: ['공급파이프라인_세대', '미분양_구', '미분양_서울', '미분양_서울_3m변화', '미분양_있음', '승인12m_대비5년평균', '승인세대_12m', '승인세대_12m_서울', '승인세대_6m', '인허가아파트_서울_12m', '인허가아파트_서울_6m', '준공12m_대비5년평균', '준공세대_12m', '준공세대_12m_서울', '준공세대_3m', '준공세대_6m', '준공아파트_서울_12m', '준공아파트_서울_3m', '준공아파트_서울_6m', '준공후미분양_구', '준공후미분양_서울']
지연 1개월로 줄였을 때 차이가 난 기준월: 10 / 10
✅ 공급 Feature: 기준월 m-2월 이후 사건 미사용, 검증이 실제 누수를 잡아냄


In [13]:
# ============================================================
# 3. 입력 지연 민감도 (2022-10 이후)
# ============================================================

recent_months = MODEL_MONTHS[MODEL_MONTHS >= pd.Period("2022-10", freq="M")]
full = supply_rm.set_index(["month", "자치구"])

rows = []
for m in recent_months:
    cutoff = m.to_timestamp()
    a = approvals[approvals["생성일"] < cutoff]
    c = completions[completions["생성일"] < cutoff]
    sub = build_supply(a, c, unsold, pd.PeriodIndex([m])).set_index(["month", "자치구"])
    for col in ["승인세대_12m", "준공세대_12m", "공급파이프라인_세대"]:
        x, y = sub[col], full.loc[sub.index, col]
        rows.append({"month": m, "feature": col,
                     "변경 행 비율": (x != y).mean(),
                     "합계 차이 비율": (y.sum() - x.sum()) / max(y.sum(), 1)})

sens = pd.DataFrame(rows)
display(sens.groupby("feature")[["변경 행 비율", "합계 차이 비율"]].agg(["mean", "max"]).round(3))

변경 행 비율       합계 차이 비율       
              mean   max     mean    max
feature                                 
공급파이프라인_세대   0.988  1.00    0.478  0.707
승인세대_12m     0.759  0.92    0.408  0.725
준공세대_12m     0.061  0.12    0.063  0.206

**결정: 승인·파이프라인 Feature는 모델에서 제외한다.**

- 2022-09 이후 승인 기록의 약 절반(세대 기준 약 3분의 1)이 승인 60일 이후에 원장에 생성됐다.
  기준일까지 생성된 기록만 쓰면 `승인세대_12m`은 행의 약 76%, `공급파이프라인_세대`는 약 99%가 달라진다.
- 사업계획승인은 고시되는 공개 행정행위라 정보 자체는 승인 시점에 공개돼 있었다.
  그러나 서비스에서 원장을 수집할 때는 최근 승인이 덜 들어와 있어, 완전한 원장으로 학습한 값과 서비스 값이 어긋난다.
- 준공(사용검사)은 차이가 작아(행 약 6%) 모델에 쓴다.
- 승인·파이프라인은 웹 '공급' 카드에 수집일 기준 현재 값으로만 보여준다.

## 5. 단변량 점검 (Train 구간만)

V1 분류 Target(향후 3개월 구 중위 평당가가 최근 3개월 대비 +3% 초과 상승)과의 단변량 AUC.
`월내_AUC`는 같은 달 안에서 구 간 차이만 본 값이다.

In [14]:
# ============================================================
# 분류 Target (V1 정의) 재현 + 단변량 AUC
# ============================================================

from sklearn.metrics import roc_auc_score

base = pd.read_parquet(PROCESSED_DIR / "df_fe_base.parquet", columns=["시군구명", "계약일자", "평당가"])
base["month"] = base["계약일자"].dt.to_period("M")
gm = (base.groupby(["시군구명", "month"])["평당가"].median().rename("med")
      .reset_index().sort_values(["시군구명", "month"]))
g = gm.groupby("시군구명")["med"]
gm["recent3"] = g.transform(lambda x: x.rolling(3, min_periods=3).median())
future = pd.concat([g.shift(-k) for k in (1, 2, 3)], axis=1)
ok = gm["recent3"].notna() & future.notna().all(axis=1)
gm["y"] = np.where(ok, (future.median(axis=1) / gm["recent3"] - 1 > 0.03).astype(float), np.nan)

tr = supply_rm.merge(gm.rename(columns={"시군구명": "자치구"})[["자치구", "month", "y"]], on=["자치구", "month"])
tr = tr[(tr["month"] < pd.Period("2025-07", freq="M")) & tr["y"].notna()]


def auc(x, y):
    m = x.notna()
    return roc_auc_score(y[m], x[m]) if x[m].nunique() > 1 else np.nan


rows = []
for c in SUPPLY_FEATURES:
    within_x = tr[c] - tr.groupby("month")[c].transform("mean")
    within_y = tr["y"] - tr.groupby("month")["y"].transform("mean")
    keep = within_y.ne(0)
    rows.append({
        "feature": c,
        "AUC": auc(tr[c], tr["y"]),
        "월내_AUC": auc(within_x[keep], within_y[keep].gt(0).astype(int)),
    })

uni = pd.DataFrame(rows)
uni["강도"] = (uni["AUC"] - 0.5).abs()
uni["용도"] = np.where(uni["feature"].isin(MODEL_SUPPLY_FEATURES), "모델", "검증용")
print("Train 행:", len(tr), "양성 비율:", round(tr["y"].mean(), 3))
display(uni.sort_values("강도", ascending=False).round(3))

Train 행: 1150 양성 비율: 0.357


,feature,AUC,월내_AUC,강도,용도
15,준공세대_12m_서울,0.400,NaN,0.100,검증용
20,준공아파트_서울_12m,0.412,NaN,0.088,모델
14,승인세대_12m_서울,0.412,NaN,0.088,검증용
17,인허가아파트_서울_12m,0.430,NaN,0.070,모델
11,미분양_서울,0.570,NaN,0.070,모델
12,준공후미분양_서울,0.562,NaN,0.062,모델
13,미분양_서울_3m변화,0.550,NaN,0.050,모델
19,준공아파트_서울_6m,0.456,NaN,0.044,모델
1,승인세대_12m,0.466,0.510,0.034,검증용
6,승인12m_대비5년평균,0.466,0.484,0.034,검증용


## 6. 웹 표시용 최신 공급 정보 (`supply_latest_v2.parquet`)

한국부동산원 입주예정물량(2026-08-31 공개분)으로 구별 향후 입주 예정 세대를 집계한다.
**모델 Feature가 아니다** (과거 회차가 없어 시점 복원 불가).

In [15]:
# ============================================================
# 입주예정물량 (최신 회차) + 최신 미분양·승인·준공
# ============================================================

ipju = pd.read_excel(IPJU_PATH)
ipju = ipju[ipju["지역"].astype(str).str.contains("서울")].copy()
ipju["자치구"] = ipju["주소"].str.extract(r"서울특별시\s+(\S+구)")[0]
# 입주예정월이 'YYYY00'(연도만 확정)인 단지는 그해 12월로 둔다
ym = ipju["입주예정월"].astype(str).str.strip()
ipju["월미정"] = ym.str.endswith("00")
ipju["입주예정월"] = pd.PeriodIndex(ym.where(~ipju["월미정"], ym.str[:4] + "12"), freq="M")
print("입주 월 미정 단지(12월로 처리):", int(ipju["월미정"].sum()))
assert ipju["자치구"].isin(ALL_GU).all()

vintage = pd.Period("2026-08", freq="M")
latest_supply = pd.DataFrame(index=ALL_GU).rename_axis("자치구")
for months in (6, 12, 24):
    within = ipju[(ipju["입주예정월"] > vintage) & (ipju["입주예정월"] <= vintage + months)]
    latest_supply[f"입주예정_{months}개월_세대"] = within.groupby("자치구")["세대수"].sum()
latest_supply["입주예정_단지수"] = ipju.groupby("자치구").size()
latest_supply = latest_supply.fillna(0).astype(int)

last_unsold = unsold[(unsold["ref_month"] == unsold["ref_month"].max()) & unsold["구"].ne("계")].set_index("구")
latest_supply["미분양"] = last_unsold["미분양"]
latest_supply["준공후미분양"] = last_unsold["준공후미분양"]
latest_supply["미분양_기준월"] = str(unsold["ref_month"].max())

# 수집일(2026-10-08) 기준 최근 12개월 승인 세대와 현재 파이프라인 (웹 표시용)
snapshot_month = pd.Period("2026-09", freq="M")
recent_apprv = approvals[approvals["승인일"].dt.to_period("M").between(snapshot_month - 11, snapshot_month)]
latest_supply["승인세대_최근12개월"] = recent_apprv.groupby("자치구")["세대수"].sum()
latest_supply["공급파이프라인_세대"] = (
    pipeline_stock(approvals, completions, pd.PeriodIndex([snapshot_month]), lag=0)
    .droplevel("month")
)
latest_supply[["승인세대_최근12개월", "공급파이프라인_세대"]] = (
    latest_supply[["승인세대_최근12개월", "공급파이프라인_세대"]].fillna(0).astype(int)
)
latest_supply["원장_수집일"] = "2026-10-08"
latest_supply["입주예정_공개일"] = "2026-08-31"
latest_supply = latest_supply.reset_index()

print("서울 입주예정:", len(ipju), "단지", int(ipju["세대수"].sum()), "세대")
display(latest_supply.sort_values("입주예정_12개월_세대", ascending=False).head(10))

입주 월 미정 단지(12월로 처리): 1
서울 입주예정: 108 단지 49668 세대


,자치구,입주예정_6개월_세대,입주예정_12개월_세대,입주예정_24개월_세대,입주예정_단지수,미분양,준공후미분양,미분양_기준월,승인세대_최근12개월,공급파이프라인_세대,원장_수집일,입주예정_공개일
14,서초구,3536,4440,10656,11,0,0,2026-08,0,5018,2026-10-08,2026-08-31
21,은평구,2451,2827,3087,5,0,0,2026-08,1802,6673,2026-10-08,2026-08-31
16,성북구,441,2332,3970,6,0,0,2026-08,256,3151,2026-10-08,2026-08-31
12,마포구,422,2261,2557,8,215,50,2026-08,1031,2094,2026-10-08,2026-08-31
15,성동구,1670,1957,4146,8,0,0,2026-08,129,3735,2026-10-08,2026-08-31
20,용산구,0,1738,1738,2,1,1,2026-08,6036,13303,2026-10-08,2026-08-31
1,강동구,0,1680,2421,5,262,262,2026-08,173,6036,2026-10-08,2026-08-31
3,강서구,901,1147,1882,9,107,107,2026-08,1395,6411,2026-10-08,2026-08-31
24,중랑구,1146,1146,1146,4,0,0,2026-08,2876,5486,2026-10-08,2026-08-31
11,동작구,914,914,4917,8,0,0,2026-08,78,14042,2026-10-08,2026-08-31


## 7. 저장

In [16]:
# ============================================================
# 구×월 저장
# - 비율 Feature(대비5년평균)는 기준 평균이 0인 구에서 결측일 수 있다.
# ============================================================

supply_v2 = (
    supply_rm.assign(기준월=supply_rm["month"].astype(str))
    .merge(region_month, on=["기준월", "자치구"], how="right")
    [["기준월", "자치구", "권역"] + MODEL_SUPPLY_FEATURES]
)

assert len(supply_v2) == 1500
assert not supply_v2.duplicated(["기준월", "자치구"]).any()

assert supply_v2[MODEL_SUPPLY_FEATURES].isna().sum().sum() == 0

supply_v2.to_parquet(PROCESSED_DIR / "supply_v2.parquet", index=False)
latest_supply.to_parquet(PROCESSED_DIR / "supply_latest_v2.parquet", index=False)

for name in ["supply_v2", "supply_latest_v2"]:
    print(name, pd.read_parquet(PROCESSED_DIR / f"{name}.parquet").shape)

supply_v2 (1500, 14)
supply_latest_v2 (25, 12)


## 정리

| 파일 | 단위 | 용도 |
|---|---|---|
| `supply_v2.parquet` | 구×월 1,500행 | 31 분류, 06 Merge — 공식 인허가·준공(서울) + 미분양(구·서울) Feature 11개 |
| `supply_latest_v2.parquet` | 구 25행 | 웹 '공급' 카드 (입주예정·미분양·최근 승인·파이프라인) |
| `supply/unsold_seoul_gu_monthly.parquet` | 구×월 | 미분양 최초 공표값 캐시 |

**남은 한계와 처리 계획**

| 한계 | 처리 |
|---|---|
| 원장은 현재 snapshot (사후 입력·승인 취소 반영 불가) | 승인·파이프라인은 입력 지연이 커서 모델 제외, 준공만 사용. 수집마다 snapshot 파일을 쌓아 이후 생성일 기준 재평가 |
| 원장 준공이 공식 통계와 불일치 (2019~2025 비교) | 서울 단위는 공식 월별 실적으로 대체, 구 단위 준공은 건축물대장(04e)으로 재구성 후 검증·반영 |
| 착공일이 2023년 이후 비어 있음 | 착공 Feature 제외. 국토부 서울 합계 착공 통계는 필요하면 서울 단위로 추가 |
| 입주예정물량은 최신 회차만 존재 | 웹 표시용. 반기마다 회차를 쌓아 이후 모델 Feature로 전환 |
| 구 재고(분모) 미반영 | 05 Demographics에서 구별 가구 수를 받은 뒤 06 Merge에서 비율화 |